# 11.0 Tableau Intelligence Layer Overview

This workbook creates the Tableau-ready dataset for the Buenavista Revenue Optimization System™.

The objective is to translate cleaned restaurant order data into an executive business intelligence layer that supports dashboard development in Tableau.

This workbook will prepare fields for:

- revenue trends
- order value segmentation
- demand by hour
- day-of-week performance
- peak vs off-peak analysis
- executive KPI reporting

The final output will be a Tableau-ready CSV file:

`orders_tableau.csv`

This dataset will power the Buenavista Revenue Optimization Dashboard and support executive decision-making around revenue growth, staffing, and operational strategy.

## 11.1 Load Clean Dataset

Load the cleaned order-level dataset created in Workbook 1.

Validate that the dataset loads successfully.

Confirm that the core fields required for Tableau dashboard development are present.

This section establishes the source dataset for the Tableau Intelligence Layer.

In [1]:
import pandas as pd

# Load cleaned Buenavista orders dataset
orders_df = pd.read_csv("../data/cleaned/orders_clean.csv")

# Preview dataset
orders_df.head()

,order_id,order_datetime,guest_count,server_name,table_name,discount_amount,order_total,tax_amount,tip_amount,gratuity_amount,year,quarter
0,1,2025-01-02 11:02:00,1,Jessica Mccullar,NaN,0.0,9.99,1.00,2.0,0.0,2025,Q1
1,2,2025-01-02 11:03:00,1,Jessica Mccullar,NaN,0.0,11.49,1.15,0.0,0.0,2025,Q1
2,6001,2025-01-02 11:10:00,1,Zachary O'Brian,4,0.0,26.99,2.69,0.0,0.0,2025,Q1
3,3,2025-01-02 11:24:00,1,Zachary O'Brian,1,0.0,36.23,3.62,20.0,0.0,2025,Q1
4,4,2025-01-02 11:53:00,1,Jessica Mccullar,NaN,0.0,39.24,3.92,0.0,0.0,2025,Q1


In [2]:
# Validate dataset shape and columns
print("Rows:", orders_df.shape[0])
print("Columns:", orders_df.shape[1])
print("\nColumn List:")
print(orders_df.columns.tolist())

Rows: 22156
Columns: 12

Column List:
['order_id', 'order_datetime', 'guest_count', 'server_name', 'table_name', 'discount_amount', 'order_total', 'tax_amount', 'tip_amount', 'gratuity_amount', 'year', 'quarter']


In [3]:
# Confirm datetime field type before transformation
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22156 entries, 0 to 22155
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         22156 non-null  int64  
 1   order_datetime   22156 non-null  object 
 2   guest_count      22156 non-null  int64  
 3   server_name      22156 non-null  object 
 4   table_name       10566 non-null  object 
 5   discount_amount  22156 non-null  float64
 6   order_total      22156 non-null  float64
 7   tax_amount       22156 non-null  float64
 8   tip_amount       22156 non-null  float64
 9   gratuity_amount  22156 non-null  float64
 10  year             22156 non-null  int64  
 11  quarter          22156 non-null  object 
dtypes: float64(5), int64(3), object(4)
memory usage: 2.0+ MB


## 11.2 Create Date Intelligence Fields

Create Tableau-friendly date fields from the order-level datetime column.

These features support:

- revenue trend analysis
- demand by hour reporting
- day-of-week performance analysis
- monthly executive reporting
- weekly trend monitoring

The goal is to enrich the transactional dataset with time-based dimensions commonly used in business intelligence dashboards.

Fields created in this section include:

- order_date
- order_hour
- order_dow
- month
- month_num
- week

These fields will enable flexible filtering, aggregation, and visualization within Tableau.

In [4]:
# Convert order_datetime to datetime format
orders_df["order_datetime"] = pd.to_datetime(
    orders_df["order_datetime"]
)

# Confirm conversion
orders_df["order_datetime"].dtype

dtype('<M8[ns]')

In [5]:
# Create Tableau date intelligence fields
orders_df["order_date"] = (
    orders_df["order_datetime"].dt.date
)

orders_df["order_hour"] = (
    orders_df["order_datetime"].dt.hour
)

orders_df["order_dow"] = (
    orders_df["order_datetime"].dt.day_name()
)

orders_df["month"] = (
    orders_df["order_datetime"].dt.month_name()
)

orders_df["month_num"] = (
    orders_df["order_datetime"].dt.month
)

orders_df["week"] = (
    orders_df["order_datetime"].dt.isocalendar().week
)

In [6]:
# Preview new date intelligence fields
orders_df[[
    "order_datetime",
    "order_date",
    "order_hour",
    "order_dow",
    "month",
    "month_num",
    "week"
]].head()

,order_datetime,order_date,order_hour,order_dow,month,month_num,week
0,2025-01-02 11:02:00,2025-01-02,11,Thursday,January,1,1
1,2025-01-02 11:03:00,2025-01-02,11,Thursday,January,1,1
2,2025-01-02 11:10:00,2025-01-02,11,Thursday,January,1,1
3,2025-01-02 11:24:00,2025-01-02,11,Thursday,January,1,1
4,2025-01-02 11:53:00,2025-01-02,11,Thursday,January,1,1


## 11.3 Create Revenue Segmentation Fields

Segment orders into meaningful revenue tiers based on total order value.

These segments help identify where revenue is generated and where opportunities for growth exist.

The segmentation framework supports:

- revenue concentration analysis
- executive KPI reporting
- Tableau dashboard filtering
- Average Order Value (AOV) optimization strategies
- identification of low-, mid-, and high-value customer behavior

Revenue segments are defined using thresholds established throughout the Buenavista Revenue Optimization System™:

- Low Value (<15)
- Mid Value (15–30)
- High Value (30+)

These categories will enable leadership to understand how different order types contribute to overall business performance.

In [8]:
# Create order value segments
orders_df["order_value_segment"] = "High Value (30+)"

orders_df.loc[
    orders_df["order_total"] <= 30,
    "order_value_segment"
] = "Mid Value (15–30)"

orders_df.loc[
    orders_df["order_total"] < 15,
    "order_value_segment"
] = "Low Value (<15)"

In [9]:
# Validate revenue segment counts
orders_df["order_value_segment"].value_counts()

order_value_segment
Mid Value (15–30)    8701
High Value (30+)     7031
Low Value (<15)      6424
Name: count, dtype: int64

In [10]:
# Validate revenue contribution by segment
orders_df.groupby("order_value_segment").agg(
    total_orders=("order_id", "count"),
    total_revenue=("order_total", "sum"),
    avg_order_value=("order_total", "mean")
).round(2)

,total_orders,total_revenue,avg_order_value
order_value_segment,,,
High Value (30+),7031,316591.06,45.03
Low Value (<15),6424,63617.09,9.90
Mid Value (15–30),8701,197083.51,22.65


## 11.4 Create Operational Intelligence Flags

Create operational classifications based on observed demand patterns.

These flags help distinguish between periods of high demand, underutilized capacity, and standard operating conditions.

The operational framework supports:

- peak-hour revenue optimization
- off-peak revenue activation strategies
- staffing alignment
- executive dashboard filtering
- identification of operational opportunities

Periods are defined using findings established throughout the Buenavista Revenue Optimization System™.

Operational classifications include:

- Peak
- Off-Peak
- Standard

These flags provide business context to transactional data and support leadership decision-making around labor deployment and revenue optimization.

In [11]:
# Default classification
orders_df["operational_period"] = "Standard"

# Peak periods
orders_df.loc[
    (
        orders_df["order_hour"].between(11, 13)
    ) |
    (
        orders_df["order_hour"].between(17, 19)
    ),
    "operational_period"
] = "Peak"

# Off-peak periods
orders_df.loc[
    orders_df["order_hour"].between(14, 16),
    "operational_period"
] = "Off-Peak"

In [12]:
# Validate operational classifications
orders_df["operational_period"].value_counts()

operational_period
Peak        16563
Off-Peak     5515
Standard       78
Name: count, dtype: int64

In [13]:
orders_df.groupby("operational_period").agg(
    total_orders=("order_id", "count"),
    total_revenue=("order_total", "sum"),
    avg_order_value=("order_total", "mean")
).round(2)

,total_orders,total_revenue,avg_order_value
operational_period,,,
Off-Peak,5515,139666.84,25.32
Peak,16563,436373.93,26.35
Standard,78,1250.89,16.04


### 11.4 Operational Intelligence Insights

Peak periods account for approximately three-quarters of all orders and revenue, confirming that operational efficiency during these windows has the greatest impact on overall business performance.

Off-Peak periods contribute roughly one-quarter of revenue while maintaining a comparable Average Order Value (AOV). This suggests that the primary opportunity during these periods lies in increasing order volume rather than increasing spend per transaction.

Standard operating periods represent a minimal share of business activity and have limited influence on overall performance.

These findings reinforce the need for differentiated strategies across operational periods, with Peak initiatives focused on throughput optimization and Off-Peak initiatives focused on demand activation.

## 11.5 Executive KPI Validation

Validate that the Tableau Intelligence Layer accurately reflects the core business insights established throughout the Buenavista Revenue Optimization System™.

This quality assurance step confirms that the engineered fields align with previous analyses and are suitable for executive dashboard development.

The objectives of this section are to:

- verify overall business performance metrics
- validate revenue segmentation outputs
- confirm operational classifications
- establish baseline KPIs for Tableau dashboards
- ensure consistency with prior workbooks

These KPIs represent the foundation of executive reporting and will be prominently featured throughout the Tableau Intelligence Layer.

In [14]:
# Executive KPI Summary
print("=== Executive KPI Summary ===\n")

print(f"Total Orders: {orders_df['order_id'].count():,}")

print(f"Total Revenue: ${orders_df['order_total'].sum():,.2f}")

print(
    f"Average Order Value: "
    f"${orders_df['order_total'].mean():,.2f}"
)

print(
    f"Average Guests per Order: "
    f"{orders_df['guest_count'].mean():.2f}"
)

=== Executive KPI Summary ===

Total Orders: 22,156
Total Revenue: $577,291.66
Average Order Value: $26.06
Average Guests per Order: 1.24


In [15]:
# Revenue segment summary
orders_df.groupby("order_value_segment").agg(
    total_orders=("order_id", "count"),
    total_revenue=("order_total", "sum")
).round(2)

,total_orders,total_revenue
order_value_segment,,
High Value (30+),7031,316591.06
Low Value (<15),6424,63617.09
Mid Value (15–30),8701,197083.51


In [16]:
# Operational period summary
orders_df.groupby("operational_period").agg(
    total_orders=("order_id", "count"),
    total_revenue=("order_total", "sum")
).round(2)

,total_orders,total_revenue
operational_period,,
Off-Peak,5515,139666.84
Peak,16563,436373.93
Standard,78,1250.89


In [17]:
# Final Tableau field check
orders_df.columns.tolist()

['order_id',
 'order_datetime',
 'guest_count',
 'server_name',
 'table_name',
 'discount_amount',
 'order_total',
 'tax_amount',
 'tip_amount',
 'gratuity_amount',
 'year',
 'quarter',
 'order_date',
 'order_hour',
 'order_dow',
 'month',
 'month_num',
 'week',
 'order_value_segment',
 'operational_period']

### 11.5 Executive KPI Insights

Validation confirms that the Tableau Intelligence Layer accurately represents the strategic findings established throughout the Buenavista Revenue Optimization System™.

Peak operating periods continue to account for the majority of orders and revenue, reinforcing the importance of throughput optimization and staffing alignment.

Revenue segmentation demonstrates that higher-value transactions contribute disproportionately to overall business performance, while mid-value orders remain the strongest opportunity for Average Order Value (AOV) expansion.

Operational classifications and engineered business intelligence fields provide a robust foundation for executive dashboard development.

These validated KPIs ensure that Tableau dashboards will communicate accurate, actionable insights capable of supporting leadership decision-making.

## 11.6 Export Tableau Dataset

Export the completed Tableau Intelligence Layer dataset for use in Tableau Public.

This dataset consolidates the engineered business intelligence fields developed throughout Workbook 11 and serves as the primary data source for executive dashboard creation.

The exported dataset supports:

- revenue trend analysis
- revenue segmentation
- demand by hour reporting
- operational period analysis
- executive KPI reporting

The resulting file will power the Buenavista Revenue Optimization Dashboard and subsequent Tableau Public assets.

Export file:

`orders_tableau.csv`

In [25]:
# Export Tableau-ready dataset
orders_df.to_csv(
    "data/cleaned/orders_tableau.csv",
    index=False
)

print("Export complete:")
print("data/cleaned/orders_tableau.csv")

Export complete:
data/cleaned/orders_tableau.csv


In [26]:
# Reload exported dataset for validation
tableau_df = pd.read_csv(
    "data/cleaned/orders_tableau.csv"
)

print("Rows:", tableau_df.shape[0])
print("Columns:", tableau_df.shape[1])

tableau_df.head()

Rows: 22156
Columns: 20


,order_id,order_datetime,guest_count,server_name,table_name,discount_amount,order_total,tax_amount,tip_amount,gratuity_amount,year,quarter,order_date,order_hour,order_dow,month,month_num,week,order_value_segment,operational_period
0,1,2025-01-02 11:02:00,1,Jessica Mccullar,NaN,0.0,9.99,1.00,2.0,0.0,2025,Q1,2025-01-02,11,Thursday,January,1,1,Low Value (<15),Peak
1,2,2025-01-02 11:03:00,1,Jessica Mccullar,NaN,0.0,11.49,1.15,0.0,0.0,2025,Q1,2025-01-02,11,Thursday,January,1,1,Low Value (<15),Peak
2,6001,2025-01-02 11:10:00,1,Zachary O'Brian,4,0.0,26.99,2.69,0.0,0.0,2025,Q1,2025-01-02,11,Thursday,January,1,1,Mid Value (15–30),Peak
3,3,2025-01-02 11:24:00,1,Zachary O'Brian,1,0.0,36.23,3.62,20.0,0.0,2025,Q1,2025-01-02,11,Thursday,January,1,1,High Value (30+),Peak
4,4,2025-01-02 11:53:00,1,Jessica Mccullar,NaN,0.0,39.24,3.92,0.0,0.0,2025,Q1,2025-01-02,11,Thursday,January,1,1,High Value (30+),Peak


In [20]:
# Final Tableau dataset fields
tableau_df.columns.tolist()

['order_id',
 'order_datetime',
 'guest_count',
 'server_name',
 'table_name',
 'discount_amount',
 'order_total',
 'tax_amount',
 'tip_amount',
 'gratuity_amount',
 'year',
 'quarter',
 'order_date',
 'order_hour',
 'order_dow',
 'month',
 'month_num',
 'week',
 'order_value_segment',
 'operational_period']

### 11.6 Tableau Export Insights

The Tableau Intelligence Layer has been successfully prepared and exported for dashboard development.

The resulting dataset integrates transactional, temporal, operational, and strategic business intelligence fields into a single executive-ready source.

This export represents the transition from analytical exploration to business intelligence deployment.

The Buenavista Revenue Optimization System™ is now equipped to support Tableau dashboard creation, Tableau Public publication, GitHub integration, LinkedIn distribution, and interview storytelling.

The next phase focuses on translating these validated insights into executive-facing visualizations capable of supporting real-world decision-making.

## 11.7 Tableau Preparation Insights

Workbook 11 successfully transformed the Buenavista transactional dataset into a Tableau-ready business intelligence asset.

Throughout this workbook, order-level data was enhanced with temporal, operational, and strategic dimensions designed to support executive dashboard development.

Key additions included:

- date intelligence fields for trend analysis
- revenue segmentation categories for performance evaluation
- operational classifications to distinguish Peak, Off-Peak, and Standard periods
- executive KPI validation to ensure consistency with prior analyses

The resulting Tableau dataset integrates the analytical findings established throughout the Buenavista Revenue Optimization System™ into a single source of truth suitable for visualization and executive reporting.

With the completion of this workbook, Buenavista transitions from analytical exploration to business intelligence deployment.

The Tableau Intelligence Layer provides the foundation for dashboard development, Tableau Public publication, GitHub integration, LinkedIn distribution, and interview storytelling.

Workbook 11 marks the beginning of Buenavista's Executive BI phase.